# Find faces in your photos (Colab)

Run the cells in order. Choose **ZIP upload** or **Drive folder**, then click **Build gallery**. For a ZIP uploaded through Colab's Files sidebar (rather than the Gallery ZIP button), enter its `/content/filename.zip` path in **ZIP path**. Upload a query photo and click **Search**. Each detected face is indexed separately; scores are cosine similarities, **not** identity probabilities. Photos stay in this Colab runtime (except Drive files you explicitly mount); the models download from the public OpenCV Zoo. Use only photos you have permission to process. The index is in memory and disappears when the runtime resets.


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-contrib-python-headless", "faiss-cpu", "pillow", "ipywidgets"])


In [ ]:
from io import BytesIO
import hashlib
from pathlib import Path
import shutil
import urllib.request
import zipfile

import cv2
import faiss
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from PIL import Image, ImageDraw
from google.colab import drive, output

output.enable_custom_widget_manager()

# Pinned OpenCV Zoo revision; media.githubusercontent.com serves the actual ONNX files, not LFS pointers.
ZOO_REV = "47534e27c9851bb1128ccc0102f1145e27f23f98"
MODELS = {
    "yunet": ("face_detection_yunet/face_detection_yunet_2023mar.onnx", "8f2383e4dd3cfbb4553ea8718107fc0423210dc964f9f4280604804ed2552fa4"),
    "sface": ("face_recognition_sface/face_recognition_sface_2021dec.onnx", "0ba9fbfa01b5270c96627c4ef784da859931e02f04419c829e83484087c34e79"),
}
def verified(path, expected):
    if not path.is_file():
        return False
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest() == expected


model_paths = {}
for name, (relative, digest) in MODELS.items():
    target = Path("/content") / Path(relative).name
    if not verified(target, digest):
        print(f"Downloading {target.name} ...", flush=True)
        url = f"https://media.githubusercontent.com/media/opencv/opencv_zoo/{ZOO_REV}/models/{relative}"
        temporary = target.with_suffix(".download")
        try:
            urllib.request.urlretrieve(url, temporary)
            if not verified(temporary, digest):
                raise ValueError(f"Model download checksum failed: {url}")
            temporary.replace(target)
        finally:
            temporary.unlink(missing_ok=True)
    model_paths[name] = str(target)

detector = cv2.FaceDetectorYN.create(model_paths["yunet"], "", (320, 320), 0.9, 0.3, 5000)
recognizer = cv2.FaceRecognizerSF.create(model_paths["sface"], "")
print("Models ready.")


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
ZIP_DIR = Path("/content/face-finder-gallery")
index = None
records = []  # One (path, display name, face box) per indexed face, in FAISS order.


def uploaded_bytes(upload):
    """Colab/ipwyidgets FileUpload value (ipywidgets 7 dict or 8 tuple)."""
    value = upload.value
    if not value:
        return None, None
    if isinstance(value, dict):
        name, item = next(iter(value.items()))
    else:
        item = value[0]
        name = item["name"]
    return name, bytes(item["content"])


def gallery_archive(upload, path):
    _, data = uploaded_bytes(upload)
    if data is None and not path.strip():
        raise ValueError("Upload a gallery ZIP with the widget, or enter the path of a ZIP uploaded through Colab's Files sidebar.")
    return zipfile.ZipFile(BytesIO(data) if data is not None else Path(path.strip()).expanduser())


def read_image(data):
    return cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_COLOR)


def faces_in(image):
    detector.setInputSize((image.shape[1], image.shape[0]))
    _, faces = detector.detect(image)
    return [] if faces is None else faces


def embedding(image, face):
    vector = recognizer.feature(recognizer.alignCrop(image, face)).reshape(1, -1).astype("float32")
    faiss.normalize_L2(vector)
    return vector


def boxed_png(image, box):
    rgb = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    x, y, w, h = map(int, box[:4])
    ImageDraw.Draw(rgb).rectangle((x, y, x + w, y + h), outline="red", width=max(2, rgb.width // 200))
    rgb.thumbnail((400, 320))
    out = BytesIO()
    rgb.save(out, format="PNG")
    return out.getvalue()


def image_widget(image, box, caption):
    return widgets.VBox([widgets.HTML(value=caption), widgets.Image(value=boxed_png(image, box), format="png")])


In [ ]:
source = widgets.RadioButtons(options=["ZIP upload", "Drive folder"], description="Gallery:")
gallery_zip = widgets.FileUpload(accept=".zip", multiple=False, description="Gallery ZIP")
zip_path = widgets.Text(value="", placeholder="/content/gallery.zip (if uploaded via Files sidebar)", description="ZIP path:", layout=widgets.Layout(width="650px"))
drive_path = widgets.Text(value="/content/drive/MyDrive/", description="Folder:", layout=widgets.Layout(width="650px"))
mount_button = widgets.Button(description="Mount Drive")
build_button = widgets.Button(description="Build gallery", button_style="primary")
query_upload = widgets.FileUpload(accept="image/*", multiple=False, description="Query image")
top_k = widgets.BoundedIntText(value=5, min=1, max=50, description="Top matches:")
threshold = widgets.FloatSlider(value=0.363, min=0.0, max=1.0, step=0.01, description="Min score:", readout_format=".2f")
search_button = widgets.Button(description="Search", button_style="success")
progress = widgets.Output()
results = widgets.Output()


def mount_drive(_):
    with progress:
        progress.clear_output()
        try:
            drive.mount("/content/drive")
        except Exception as exc:
            print(f"Drive mount failed: {exc}")


def build_gallery(_):
    global index, records
    # Never search a stale index after a failed rebuild.
    index, records = None, []
    results.clear_output()
    with progress:
        progress.clear_output()
        try:
            paths = []
            if source.value == "ZIP upload":
                with gallery_archive(gallery_zip, zip_path.value) as archive:
                    entries = [info for info in archive.infolist() if not info.is_dir() and Path(info.filename).suffix.lower() in IMAGE_SUFFIXES]
                    if not entries:
                        raise ValueError("The ZIP contains no supported images.")
                    if sum(info.file_size for info in entries) > 1_000_000_000 or any(info.file_size > 30_000_000 for info in entries):
                        raise ValueError("ZIP images exceed the 1 GB total / 30 MB per image safety limit.")
                    shutil.rmtree(ZIP_DIR, ignore_errors=True)
                    ZIP_DIR.mkdir(parents=True)
                    # Generated names: never extract untrusted ZIP paths onto disk.
                    for n, info in enumerate(entries):
                        path = ZIP_DIR / f"{n:06d}{Path(info.filename).suffix.lower()}"
                        with archive.open(info) as src, path.open("wb") as dst:
                            shutil.copyfileobj(src, dst)
                        paths.append((path, info.filename))
            else:
                folder = Path(drive_path.value.strip()).expanduser()
                if not folder.is_dir():
                    raise ValueError("Drive folder not found. Mount Drive and enter an existing folder path.")
                paths = [(p, str(p.relative_to(folder))) for p in sorted(folder.rglob("*")) if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
                if not paths:
                    raise ValueError("No supported images in this folder.")

            print(f"Processing {len(paths)} images ...", flush=True)
            vectors, found, no_face, unreadable = [], [], 0, 0
            for n, (path, label) in enumerate(paths, 1):
                image = cv2.imread(str(path))
                if image is None:
                    unreadable += 1
                    continue
                faces = faces_in(image)
                if not len(faces):
                    no_face += 1
                for face in faces:
                    vectors.append(embedding(image, face)[0])
                    found.append((path, label, face[:4].astype(int)))
                if n % 10 == 0 or n == len(paths):
                    print(f"{n}/{len(paths)} images, {len(found)} faces", flush=True)
            print(f"Done: {len(found)} faces; {no_face} images without faces; {unreadable} unreadable images.")
            if not vectors:
                print("No faces to index. Try different gallery images.")
                return
            matrix = np.ascontiguousarray(np.vstack(vectors), dtype="float32")
            index = faiss.IndexFlatIP(matrix.shape[1])
            index.add(matrix)
            records = found
            print("Gallery ready. Upload a query image, then click Search.")
        except (ValueError, OSError, zipfile.BadZipFile, RuntimeError) as exc:
            print(f"Could not build gallery: {exc}")


def search(_):
    with results:
        results.clear_output()
        if index is None:
            print("Build a gallery with faces first.")
            return
        _, data = uploaded_bytes(query_upload)
        if data is None:
            print("Upload a query image first.")
            return
        image = read_image(data)
        if image is None:
            print("Could not read the query image. Upload a supported image.")
            return
        faces = faces_in(image)
        if not len(faces):
            print("No faces detected in the query image.")
            return
        print(f"Found {len(faces)} query face(s). Showing up to {top_k.value} gallery faces per query face.")
        for number, face in enumerate(faces, 1):
            print(f"Query face {number}")
            display(image_widget(image, face, "Query"))
            scores, ids = index.search(embedding(image, face), min(top_k.value, index.ntotal))
            shown = 0
            for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), 1):
                if score < threshold.value:
                    continue
                path, label, box = records[int(idx)]
                shown += 1
                match = cv2.imread(str(path))
                if match is None:
                    print(f"#{rank}: {label} — image no longer readable")
                    continue
                display(image_widget(match, box, f"#{rank} · {label} · similarity {score:.3f}"))
            if not shown:
                print("No matches above minimum score.")


mount_button.on_click(mount_drive)
build_button.on_click(build_gallery)
search_button.on_click(search)
display(widgets.VBox([
    widgets.HTML("<h3>1. Build a gallery</h3>"), source, gallery_zip, zip_path,
    widgets.HBox([mount_button, drive_path]), build_button, progress,
    widgets.HTML("<h3>2. Search by face</h3>"), query_upload, top_k, threshold, search_button, results,
]))
